# Hybrid Quantum-Classical Reinforcement Learning for Grid World Navigation
### Course Project: ELEC/PHYS 450/550 - Spring 2026
**Presenter**: Barış Cem Bakay  
**Institution**: Koç University  
**Instructor**: Assoc. Prof. Mehmet Cengiz Onbaşlı

---

This Jupyter Notebook provides a self-contained, step-by-step implementation of the **Quantum Reinforcement Learning (QRL)** agent based on a **Variational Quantum Circuit (VQC)** policy network. 

### Key Highlights:
1. **Environment**: A custom discrete Grid World MDP modeled under Gymnasium standards.
2. **State Encoding**: 2D state coordinates duplicated and encoded via Angle Encoding ($R_y$ rotations).
3. **Trainable Ansatz**: Qiskit `RealAmplitudes` ansatz with CNOT entangling gates (16 trainable parameters).
4. **Classical Post-Processing**: PyTorch Linear output layer (20 trainable parameters) yielding a hybrid total of **36 parameters**.
5. **Physical Tomography**: Exact Quantum State Tomography to verify density matrix reconstruction, purity ($\gamma=1.0$), and Von Neumann entropy ($S=0.0$).

In [ ]:
# Install dependencies for Google Colab environment
!pip install "qiskit>=1.0.0" "qiskit-machine-learning>=0.7.2" "qiskit-aer>=0.13.0" gymnasium torch matplotlib pandas

## 1. Environment Design: Gymnasium Grid World
We model the Grid World navigation task as a discrete Markov Decision Process (MDP). 
- **State**: $(x, y)$ coordinate coordinate vector.
- **Actions**: `0: Up`, `1: Down`, `2: Left`, `3: Right`.
- **Rewards**: $+1.0$ at Goal state, $-1.0$ at Obstacle state $(1, 1)$, and step penalty of $-0.04$ per step.

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class GridWorldEnv(gym.Env):
    def __init__(self, size=3):
        super(GridWorldEnv, self).__init__()
        self.size = size
        self.observation_space = spaces.Box(low=0, high=size-1, shape=(2,), dtype=np.int32)
        self.action_space = spaces.Discrete(4)
        
        self.start_pos = np.array([0, 0])
        self.goal_pos = np.array([size - 1, size - 1])
        self.obstacle_pos = np.array([1, 1])
        self.state = self.start_pos.copy()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = self.start_pos.copy()
        return self.state, {}

    def step(self, action):
        move = np.array([0, 0])
        if action == 0:   # Up
            move = np.array([-1, 0])
        elif action == 1: # Down
            move = np.array([1, 0])
        elif action == 2: # Left
            move = np.array([0, -1])
        elif action == 3: # Right
            move = np.array([0, 1])
            
        new_state = self.state + move
        new_state[0] = np.clip(new_state[0], 0, self.size - 1)
        new_state[1] = np.clip(new_state[1], 0, self.size - 1)
        self.state = new_state
        
        done = False
        reward = -0.04
        
        if np.array_equal(self.state, self.goal_pos):
            reward = 1.0
            done = True
        elif np.array_equal(self.state, self.obstacle_pos):
            reward = -1.0
            done = True
            
        return self.state, reward, done, False, {}

    def render(self):
        grid = np.zeros((self.size, self.size), dtype=str)
        grid[:] = "."
        grid[self.start_pos[0], self.start_pos[1]] = "S"
        grid[self.goal_pos[0], self.goal_pos[1]] = "G"
        grid[self.obstacle_pos[0], self.obstacle_pos[1]] = "X"
        grid[self.state[0], self.state[1]] = "A"
        
        print("\n".join([" ".join(row) for row in grid]))
        print("-" * 10)

## 2. Quantum Policy Circuit Generation
We create a parameter-dependent quantum circuit in Qiskit consisting of:
1. **Angle Encoding**: Coordinates $(x, y)$ mapped to rotational parameters on qubits 0 and 2 ($x$), and 1 and 3 ($y$).
2. **RealAmplitudes ansatz**: Linear entangling CNOT layout across $R=3$ repetitions (totaling 16 parameters).

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import real_amplitudes

def create_qrl_circuit(num_qubits=4, layers=3):
    qc = QuantumCircuit(num_qubits)
    
    # Input parameters for normalized state coordinates (x, y)
    input_params = ParameterVector('input', 2)
    
    # State coordinates angle encoding with qubit duplication
    qc.ry(input_params[0], 0)
    qc.ry(input_params[1], 1)
    qc.ry(input_params[0], 2)
    qc.ry(input_params[1], 3)
    
    qc.barrier()
    
    # Trainable RealAmplitudes ansatz
    ansatz = real_amplitudes(num_qubits, entanglement='linear', reps=layers)
    qc.compose(ansatz, inplace=True)
    
    return qc, input_params

# Draw the circuit to verify layout
circuit, inputs = create_qrl_circuit(num_qubits=4, layers=3)
print(circuit.draw('text'))

## 3. Quantum-Classical Hybrid Policy Network (EstimatorQNN)
Using Qiskit Machine Learning's `EstimatorQNN` and PyTorch's `TorchConnector`, we define the hybrid network. Expectation values $\langle Z_i \rangle$ are scaled using a classical linear layer.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

class QuantumQNetwork(nn.Module):
    def __init__(self, grid_size=3):
        super(QuantumQNetwork, self).__init__()
        self.grid_size = grid_size
        self.circuit, self.input_params = create_qrl_circuit(num_qubits=4, layers=3)
        
        all_params = list(self.circuit.parameters)
        self.weight_params = [p for p in all_params if p not in self.input_params]
        
        # Action observables: measurement of Pauli-Z on each qubit
        self.observables = [
            SparsePauliOp.from_list([("IIIZ", 1)]),  # Action 0: Up (q0)
            SparsePauliOp.from_list([("IIZI", 1)]),  # Action 1: Down (q1)
            SparsePauliOp.from_list([("IZII", 1)]),  # Action 2: Left (q2)
            SparsePauliOp.from_list([("ZIII", 1)])   # Action 3: Right (q3)
        ]
        
        self.qnn = EstimatorQNN(
            circuit=self.circuit,
            observables=self.observables,
            input_params=self.input_params,
            weight_params=self.weight_params
        )
        
        # Wrap QNN into PyTorch layer and define classical linear scaling layer
        self.qnn_connector = TorchConnector(self.qnn)
        self.output_layer = nn.Linear(4, 4)
        
        # Initialize small weights for stable starting Q-values
        nn.init.uniform_(self.output_layer.weight, -0.1, 0.1)
        nn.init.constant_(self.output_layer.bias, 0.0)
        
    def forward(self, x):
        # Scale Cartesian state to fit into [0, pi] for encoding
        x_scaled = x * (np.pi / (self.grid_size - 1.0))
        qnn_out = self.qnn_connector(x_scaled)
        return self.output_layer(qnn_out)

## 4. Replay Buffer & Agent Logic

In [ ]:
import random
from collections import deque

class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)
    
    def __len__(self):
        return len(self.buffer)

class QuantumDQNAgent:
    def __init__(self, grid_size=3):
        self.action_dim = 4
        self.policy_net = QuantumQNetwork(grid_size=grid_size)
        self.target_net = QuantumQNetwork(grid_size=grid_size)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=0.001)
        self.memory = ReplayBuffer(5000)
        
        self.gamma = 0.99
        self.epsilon = 1.0
        self.epsilon_decay = 0.995
        self.epsilon_min = 0.01
        self.batch_size = 16
        self.target_update = 10

    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randrange(self.action_dim)
        with torch.no_grad():
            state = torch.FloatTensor(state).unsqueeze(0)
            return self.policy_net(state).argmax().item()

    def train(self):
        if len(self.memory) < self.batch_size:
            return
        
        transitions = self.memory.sample(self.batch_size)
        batch = list(zip(*transitions))
        
        state_batch = torch.FloatTensor(np.array(batch[0]))
        action_batch = torch.LongTensor(batch[1]).unsqueeze(1)
        reward_batch = torch.FloatTensor(batch[2]).unsqueeze(1)
        next_state_batch = torch.FloatTensor(np.array(batch[3]))
        done_batch = torch.FloatTensor(batch[4]).unsqueeze(1)

        current_q = self.policy_net(state_batch).gather(1, action_batch)
        
        with torch.no_grad():
            next_q = self.target_net(next_state_batch).max(1)[0].unsqueeze(1)
            expected_q = reward_batch + (1 - done_batch) * self.gamma * next_q

        loss = nn.MSELoss()(current_q, expected_q)
        
        self.optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping to stabilize policy parameter learning
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), max_norm=1.0)
        self.optimizer.step()

## 5. Training Loop
We train the Quantum DQN agent on the 3x3 Grid World. The learning process on simulator takes approximately 10-15 minutes.

In [ ]:
def train_quantum_dqn(episodes=150, grid_size=3):
    env = GridWorldEnv(size=grid_size)
    agent = QuantumDQNAgent(grid_size=grid_size)
    rewards_history = []

    print("--- Starting Quantum DQN Training ---")
    for episode in range(episodes):
        state, _ = env.reset()
        total_reward = 0
        done = False
        step_count = 0
        
        while not done and step_count < 50:
            action = agent.select_action(state)
            next_state, reward, done, _, _ = env.step(action)
            agent.memory.push(state, action, reward, next_state, done)
            agent.train()
            
            state = next_state
            total_reward += reward
            step_count += 1
        
        if episode % agent.target_update == 0:
            agent.target_net.load_state_dict(agent.policy_net.state_dict())
            
        agent.epsilon = max(agent.epsilon_min, agent.epsilon * agent.epsilon_decay)
        rewards_history.append(total_reward)
        
        if episode % 10 == 0:
            print(f"Episode {episode:3d} | Total Reward: {total_reward:6.2f} | Epsilon: {agent.epsilon:.3f}")
            
    return agent, rewards_history

# Run training for a quick demo of 120 episodes
agent, history = train_quantum_dqn(episodes=120)

## 6. Visualization & Test Run

In [ ]:
import matplotlib.pyplot as plt

# Plot reward history
plt.figure(figsize=(10, 5))
plt.plot(history, label='Quantum DQN Reward', color='darkred', linewidth=2)
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('Quantum DQN Training Convergence on 3x3 Grid')
plt.axhline(y=0.88, color='green', linestyle='--', label='Theoretical Optimal Reward (0.88)')
plt.legend()
plt.grid(True)
plt.show()

# Run a final test navigation
print("\n--- Running Verification Navigation ---")
env = GridWorldEnv()
state, _ = env.reset()
env.render()
done = False
steps = 0
while not done and steps < 15:
    with torch.no_grad():
        state_tensor = torch.FloatTensor(state).unsqueeze(0)
        q_values = agent.policy_net(state_tensor)
        action = q_values.argmax().item()
    state, reward, done, _, _ = env.step(action)
    env.render()
    steps += 1

## 7. Quantum State Tomography
To physically analyze the state of the quantum policy network, we perform exact density matrix reconstruction.

In [ ]:
from qiskit.quantum_info import DensityMatrix, purity, entropy
from qiskit_aer import AerSimulator

test_positions = [(0, 0), (1, 0), (2, 1), (2, 2)]
results = []

for pos in test_positions:
    state_tensor = torch.FloatTensor(np.array(pos)).unsqueeze(0)
    x_scaled = state_tensor * (np.pi / 2.0)
    weights = agent.policy_net.qnn_connector.weight.detach().numpy()
    
    # Configure state circuit with optimized weights
    qc = agent.policy_net.circuit.assign_parameters({
        agent.policy_net.input_params[0]: x_scaled[0, 0].item(),
        agent.policy_net.input_params[1]: x_scaled[0, 1].item()
    })
    
    param_dict = {p: weights[i] for i, p in enumerate(agent.policy_net.weight_params)}
    qc = qc.assign_parameters(param_dict)
    
    qc.save_density_matrix()
    sim = AerSimulator()
    job = sim.run(qc)
    rho = job.result().data()['density_matrix']
    
    p_val = purity(rho)
    s_val = entropy(rho)
    
    print(f"Coordinate: {pos} | Purity (gamma): {p_val:.4f} | Von Neumann Entropy (S): {s_val:.4f}")
    results.append((pos, rho, s_val))

# Plot Density Matrix Real Components
fig, axes = plt.subplots(1, len(test_positions), figsize=(18, 5))
for i, (pos, rho, s_val) in enumerate(results):
    axes[i].set_title(f"Position: {pos}\nEntropy S={s_val:.2f}")
    im = axes[i].imshow(np.real(rho.data), cmap='viridis')
    plt.colorbar(im, ax=axes[i])
plt.suptitle("Density Matrices Heatmap (Purity Verification)", y=1.05, fontsize=16)
plt.tight_layout()
plt.show()